# fase_1 - script_afrida Migration

This notebook handles migration of database from old DB to new DB for fase 1.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [4]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [29]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


## 2. Ambil Data dari DB Lama

In [6]:
# TODO: Ganti query sesuai dengan tabel yang akan dimigrasikan
query_old = """SELECT * FROM catatan_kelas LIMIT 10"""

cursor_old.execute(query_old)
data_old = cursor_old.fetchall()

print(f"Total records from old DB: {{len(data_old)}}")
print(f"Sample data: {{data_old[:3] if data_old else 'No data'}}")

Total records from old DB: {len(data_old)}
Sample data: {data_old[:3] if data_old else 'No data'}


## 3. Helper

In [7]:
def fetch_df(query):
    return pd.read_sql(query, db_old)

def check_nulls(df):
    print("\nNULL CHECK:")
    display(df.isnull().sum())

def check_duplicates(df, subset_cols):
    dup = df[df.duplicated(subset=subset_cols)]
    print(f"\nDUPLICATE ROWS: {len(dup)}")
    display(dup.head())

def preview_df(df, title="Preview", limit=10):
    print(f"\n{title}:")
    display(df.head(limit))

def compare_count(table_old, table_new):
    old = pd.read_sql(f"SELECT COUNT(*) as total FROM {table_old}", db_old)['total'][0]
    new = pd.read_sql(f"SELECT COUNT(*) as total FROM {table_new}", db_new)['total'][0]

    print(f"\nCOUNT CHECK → OLD: {old} | NEW: {new}")
    print("STATUS:", "OK ✅" if old == new else "CHECK ⚠️")

def check_dtype(df, table_name):
    print(f"\n=== CHECK TIPE DATA: {table_name} ===")

    db_schema = pd.read_sql(f"DESCRIBE {table_name}", db_new)

    for col in df.columns:
        df_type = df[col].dtype

        db_type = db_schema[db_schema['Field'] == col]['Type'].values
        db_type = db_type[0] if len(db_type) > 0 else "NOT FOUND"

        print(f"{col} → DF: {df_type} | DB: {db_type}")

In [8]:
def report_not_null_violations(df, table_name):
    print(f"\n=== NOT NULL VIOLATION: {table_name} ===")

    schema = pd.read_sql(f"DESCRIBE {table_name}", db_new)

    violations = {}

    for _, row in schema.iterrows():
        col = row['Field']
        is_nullable = row['Null']

        if col in df.columns and is_nullable == 'NO':
            null_count = df[col].isnull().sum()

            if null_count > 0:
                violations[col] = null_count

    if not violations:
        print("✅ Semua kolom NOT NULL aman")
        return False
    else:
        print("⚠️ Kolom NOT NULL yang bermasalah:")
        for k, v in violations.items():
            print(f"{k}: {v} NULL")
        return True
    
def enforce_not_null(df, table_name):
    schema = pd.read_sql(f"DESCRIBE {table_name}", db_new)

    for _, row in schema.iterrows():
        col = row['Field']
        is_nullable = row['Null']
        col_type = row['Type']

        if col in df.columns and is_nullable == 'NO':
            if df[col].isnull().sum() > 0:

                if 'int' in col_type:
                    df[col] = df[col].fillna(0)
                elif 'date' in col_type or 'time' in col_type:
                    df[col] = df[col].fillna(pd.Timestamp('1970-01-01'))
                else:
                    df[col] = df[col].fillna('Unknown')

    return df

## 4. Migrating

In [9]:
def migrate_kursus():
    print("\n=== MIGRATING KURSUS ===")

    # 1. EXTRACT
df_old = fetch_df("""
    SELECT idpendkursus, nama_kursus, keterangan
    FROM pendidikankursus
""")

preview_df(df_old, "DATA ASLI")

# 2. TRANSFORM (mapping kolom)
df = df_old.rename(columns={
    'idpendkursus': 'id_kursus',
    'nama_kursus': 'nama_kursus',
    'keterangan': 'deskripsi'
})

# 3. CLEANING

## handle null
df['deskripsi'] = df['deskripsi'].fillna('')

## pastikan tipe
df['id_kursus'] = df['id_kursus'].astype(str)
check_dtype(df, "kursus")

# 4. VALIDASI
check_nulls(df)
check_duplicates(df, ['id_kursus'])

# 5. PREVIEW HASIL TRANSFORM
preview_df(df, "SETELAH TRANSFORM")


DATA ASLI:


,idpendkursus,nama_kursus,keterangan
0,K00001,LEAP - General English,"GE, Balloons, Gogo, SO, Winner"
1,K00002,LEAP - Coding Class,Coding Class Regular
2,K00003,LEAP - Leap Literacy Club,LLC
3,K00004,LEAP - Conversation Class,Conversation Class for Adults
4,K00005,LEAP - Aplikasi Perkantoran,"All In, Private,"
5,K00006,Kemitraan - B2B Supervisi Guru,Nurul Faizah
6,K00007,Kemitraan - B2B Business English,"Delta Jaya, Hartono, PT HCA"
7,K00009,Kemitraan - B2C Talentvis,Talentvis
8,K00010,LEAP - General English 2024,NEW CURRICULUM 2024
9,K00011,Kemitraan - B2B Language Upskilling Program,Nurul Faizah Agt 24 - Mei 25



=== CHECK TIPE DATA: kursus ===
id_kursus → DF: str | DB: varchar(15)
nama_kursus → DF: str | DB: varchar(150)
deskripsi → DF: str | DB: text

NULL CHECK:


id_kursus      0
nama_kursus    0
deskripsi      0
dtype: int64


DUPLICATE ROWS: 0


,id_kursus,nama_kursus,deskripsi



SETELAH TRANSFORM:


,id_kursus,nama_kursus,deskripsi
0,K00001,LEAP - General English,"GE, Balloons, Gogo, SO, Winner"
1,K00002,LEAP - Coding Class,Coding Class Regular
2,K00003,LEAP - Leap Literacy Club,LLC
3,K00004,LEAP - Conversation Class,Conversation Class for Adults
4,K00005,LEAP - Aplikasi Perkantoran,"All In, Private,"
5,K00006,Kemitraan - B2B Supervisi Guru,Nurul Faizah
6,K00007,Kemitraan - B2B Business English,"Delta Jaya, Hartono, PT HCA"
7,K00009,Kemitraan - B2C Talentvis,Talentvis
8,K00010,LEAP - General English 2024,NEW CURRICULUM 2024
9,K00011,Kemitraan - B2B Language Upskilling Program,Nurul Faizah Agt 24 - Mei 25


In [ ]:
def migrate_level():
    print("\n=== MIGRATING level ===")

    # 1. EXTRACT
df_old = fetch_df("""
        SELECT idlevel, level, tingkatan
        FROM level
    """)

preview_df(df_old, "DATA ASLI")

# 2. TRANSFORM (mapping kolom)
df = df_old.rename(columns={
    'idlevel': 'id_level',
        'level': 'nama_level',
        'tingkatan': 'urutan_level'
})

# 3. CLEANING

## handle null
df['nama_level'] = df['nama_level'].fillna('Unknown')

# cek pelanggaran NOT NULL
violation = report_not_null_violations(df, "level")

## pastikan tipe
df['id_level'] = df['id_level'].astype(str)
check_dtype(df, "level")

# 4. VALIDASI
check_nulls(df)
check_duplicates(df, ['id_level'])

# 5. PREVIEW HASIL TRANSFORM
preview_df(df, "SETELAH TRANSFORM")


In [11]:
df[df['urutan_level'] % 1 != 0]
df['urutan_level'] = df['urutan_level'].fillna(0).astype(int)

In [12]:
check_dtype(df, "level")


=== CHECK TIPE DATA: level ===
id_level → DF: str | DB: varchar(15)
nama_level → DF: str | DB: varchar(100)
urutan_level → DF: int64 | DB: int(11)


In [13]:
def migrate_libur():
    print("\n=== MIGRATING LIBUR ===")

    # 1. EXTRACT
df_old = fetch_df("""
        SELECT idlibur, title, start, end
        FROM libur
    """)

preview_df(df_old, "DATA ASLI")

# 2. TRANSFORM (mapping kolom)
df = df_old.rename(columns={
    'idlibur': 'id_libur',
    'title': 'nama_event',
    'start': 'tanggal_mulai',
    'end': 'tanggal_berakhir'
})

# 3. CLEANING

## handle null
df['nama_event'] = df['nama_event'].fillna('Tidak Ada')

# cek pelanggaran NOT NULL
violation = report_not_null_violations(df, "libur")
def migrate_libur():

    if violation:
        print("\n APPLY ENFORCE NOT NULL")
        df = enforce_not_null(df, "libur")

# 8. DUPLICATE
check_duplicates(df, ['id_libur'])

# 9. PREVIEW
preview_df(df, "SETELAH TRANSFORM")

## pastikan tipe
df['id_libur'] = df['id_libur'].astype(str)
check_dtype(df, "libur")

# 4. VALIDASI
check_nulls(df)
check_duplicates(df, ['id_libur'])

# 5. PREVIEW HASIL TRANSFORM
preview_df(df, "SETELAH TRANSFORM")



DATA ASLI:


,idlibur,title,start,end
0,L00005,Libur Nasional,2023-07-19,2023-07-20
1,L00006,Libur Nasional,2023-06-29,2023-06-30
2,L00007,Tahun Baru Islam 2023,2023-07-19,2023-07-20
3,L00008,Natal,2023-12-22,2023-12-30
4,L00009,Tahun Baru,2024-01-01,2024-01-02
5,L00010,Isra' Miraj,2024-02-08,2024-02-09
6,L00011,Nyepi,2024-03-11,2024-03-12
7,L00012,Hari Kemerdekaan,2023-08-17,2023-08-18
8,L00013,Maulid Nabi Muhammad SAW,2023-09-28,2023-09-29
9,L00014,Isra Mi'raj,2024-02-08,2024-02-09



=== NOT NULL VIOLATION: libur ===
✅ Semua kolom NOT NULL aman

DUPLICATE ROWS: 0


,id_libur,nama_event,tanggal_mulai,tanggal_berakhir



SETELAH TRANSFORM:


...

id_libur            0
nama_event          0
tanggal_mulai       0
tanggal_berakhir    0
dtype: int64


DUPLICATE ROWS: 0


,id_libur,nama_event,tanggal_mulai,tanggal_berakhir



SETELAH TRANSFORM:


,id_libur,nama_event,tanggal_mulai,tanggal_berakhir
0,L00005,Libur Nasional,2023-07-19,2023-07-20
1,L00006,Libur Nasional,2023-06-29,2023-06-30
2,L00007,Tahun Baru Islam 2023,2023-07-19,2023-07-20
3,L00008,Natal,2023-12-22,2023-12-30
4,L00009,Tahun Baru,2024-01-01,2024-01-02
5,L00010,Isra' Miraj,2024-02-08,2024-02-09
6,L00011,Nyepi,2024-03-11,2024-03-12
7,L00012,Hari Kemerdekaan,2023-08-17,2023-08-18
8,L00013,Maulid Nabi Muhammad SAW,2023-09-28,2023-09-29
9,L00014,Isra Mi'raj,2024-02-08,2024-02-09


In [14]:
df['tanggal_mulai'] = pd.to_datetime(df['tanggal_mulai'], errors='coerce')
df['tanggal_berakhir'] = pd.to_datetime(df['tanggal_berakhir'], errors='coerce')

df.dtypes

id_libur                      str
nama_event                    str
tanggal_mulai       datetime64[s]
tanggal_berakhir    datetime64[s]
dtype: object

In [15]:
check_dtype(df, "libur")


=== CHECK TIPE DATA: libur ===
id_libur → DF: str | DB: varchar(20)
nama_event → DF: str | DB: varchar(150)
tanggal_mulai → DF: datetime64[s] | DB: date
tanggal_berakhir → DF: datetime64[s] | DB: date


In [28]:
def migrate_topik_diskusi():
    print("\n=== MIGRATING TOPIK DISKUSI ===")

    # 1. EXTRACT
df_old = fetch_df("""
        SELECT idtagmd, tag
        FROM tag_materi_diskusi
    """)

preview_df(df_old, "DATA ASLI")

# 2. TRANSFORM (mapping kolom)
df = df_old.rename(columns={
    'idtagmd': 'id_topik_diskusi',
    'tag': 'topik_diskusi'
})

# opsional: kosongin biar jelas tidak dipakai
df['id_topik_diskusi'] = None

# 3. CLEANING

## handle null
df['topik_diskusi'] = df['topik_diskusi'].fillna('Tidak Ada')


# optional: buang spasi aneh
df['topik_diskusi'] = df['topik_diskusi'].str.strip()

# 4. REMOVE DUPLICATE (penting buat auto increment)
df = df.drop_duplicates(subset=['topik_diskusi'])

# cek pelanggaran NOT NULL
violation = report_not_null_violations(df, "topik_diskusi")
def migrate_topik_diskusi():

    if violation:
        print("\n APPLY ENFORCE NOT NULL")
        df = enforce_not_null(df, "topik_diskusi")

# 8. DUPLICATE
check_duplicates(df, ['id_topik_diskusi'])

# 9. PREVIEW
preview_df(df, "SETELAH TRANSFORM")

## pastikan tipe
df['id_topik_diskusi'] = df['id_topik_diskusi'].astype(str)
check_dtype(df, "topik_diskusi")

# 4. VALIDASI
check_nulls(df)
check_duplicates(df, ['id_topik_diskusi'])

# 5. PREVIEW HASIL TRANSFORM
preview_df(df, "SETELAH TRANSFORM")



DATA ASLI:


,idtagmd,tag
0,T00003,Kendala Siswa
1,T00004,Kendala Kelas
2,T00005,Kendala Jadwal
3,T00006,Ujian Susulan & Remidi
4,T00007,"Kendala Zoom, Class In, Koneksi & Device"
5,T00008,Update Diskusi
6,T00009,Siswa Off/Postponed/Pindah Program
7,T00010,Progress Siswa
8,T00011,Update Siswa Sit-in/Trial/Baru
9,T00012,Siswa Tidak Naik



=== NOT NULL VIOLATION: topik_diskusi ===
⚠️ Kolom NOT NULL yang bermasalah:
id_topik_diskusi: 11 NULL

DUPLICATE ROWS: 10


,id_topik_diskusi,topik_diskusi
1,None,Kendala Kelas
2,None,Kendala Jadwal
3,None,Ujian Susulan & Remidi
4,None,"Kendala Zoom, Class In, Koneksi & Device"
5,None,Update Diskusi



SETELAH TRANSFORM:


...

id_topik_diskusi    11
topik_diskusi        0
dtype: int64


DUPLICATE ROWS: 10


,id_topik_diskusi,topik_diskusi
1,NaN,Kendala Kelas
2,NaN,Kendala Jadwal
3,NaN,Ujian Susulan & Remidi
4,NaN,"Kendala Zoom, Class In, Koneksi & Device"
5,NaN,Update Diskusi



SETELAH TRANSFORM:


,id_topik_diskusi,topik_diskusi
0,NaN,Kendala Siswa
1,NaN,Kendala Kelas
2,NaN,Kendala Jadwal
3,NaN,Ujian Susulan & Remidi
4,NaN,"Kendala Zoom, Class In, Koneksi & Device"
5,NaN,Update Diskusi
6,NaN,Siswa Off/Postponed/Pindah Program
7,NaN,Progress Siswa
8,NaN,Update Siswa Sit-in/Trial/Baru
9,NaN,Siswa Tidak Naik


In [17]:
df = enforce_not_null(df, "topik_diskusi")

In [18]:
violation = report_not_null_violations(df, "topik_diskusi")


=== NOT NULL VIOLATION: topik_diskusi ===
✅ Semua kolom NOT NULL aman


In [45]:
def migrate_kursus_level():
    print("\n=== MIGRATING KURSUS LEVEL ===")

    # 1. EXTRACT
df_old = fetch_df("""
        SELECT idpendkursus, idlevel
        FROM level
    """)

preview_df(df_old, "DATA ASLI")

# 2. TRANSFORM (mapping kolom)
df = df_old.rename(columns={
        'idpendkursus': 'id_kursus',
        'idlevel': 'id_level'
    })

# opsional: kosongin biar jelas tidak dipakai
df['id_topik_diskusi'] = None

# 3. CLEANING

df['id_kursus'] = df['id_kursus'].astype(str)
df['id_level'] = pd.to_numeric(df['id_level'], errors='coerce')

## pastikan tipe
check_dtype(df, "kursus_level")

# 4. VALIDASI
check_nulls(df)
check_duplicates(df, ['id_kursus'])

    # 4. REMOVE DUPLICATE (relasi sering dobel)
df = df.drop_duplicates(subset=['id_kursus', 'id_level'])

   # 5. NULL CHECK
report_not_null_violations(df, "kursus_level")

    # 6. VALIDASI FK (opsional tapi bagus)
    # cek apakah id ada di tabel master
df_kursus = fetch_df("SELECT id_kursus FROM kursus_level")
df_level = fetch_df("SELECT id_level FROM level")

df = df[df['id_kursus'].isin(df_kursus['id_kursus'])]
df = df[df['id_level'].isin(df_level['id_level'])]

    # 7. PREVIEW
preview_df(df, "SETELAH TRANSFORM")




DATA ASLI:


,idpendkursus,idlevel
0,K00001,L00001
1,K00001,L00002
2,K00001,L00003
3,K00001,L00004
4,K00001,L00005
5,K00001,L00006
6,K00001,L00007
7,K00001,L00008
8,K00001,L00009
9,K00001,L00011



=== CHECK TIPE DATA: kursus_level ===
id_kursus → DF: str | DB: varchar(20)
id_level → DF: float64 | DB: varchar(20)
id_topik_diskusi → DF: object | DB: NOT FOUND

NULL CHECK:


id_kursus             0
id_level            181
id_topik_diskusi    181
dtype: int64


DUPLICATE ROWS: 162


,id_kursus,id_level,id_topik_diskusi
1,K00001,NaN,None
2,K00001,NaN,None
3,K00001,NaN,None
4,K00001,NaN,None
5,K00001,NaN,None



=== NOT NULL VIOLATION: kursus_level ===
⚠️ Kolom NOT NULL yang bermasalah:
id_level: 19 NULL


ProgrammingError: 1146 (42S02): Table 'dataleap_v5_example.kursus_level' doesn't exist

In [36]:
def migrate_kursus_level():
    print("\n=== MIGRATING KURSUS LEVEL ===")

    # 1. EXTRACT
    df_old = fetch_df("""
        SELECT idpendkursus, idlevel
        FROM level
    """)

    preview_df(df_old, "DATA ASLI")

    # 2. TRANSFORM
    df = df_old.rename(columns={
        'idpendkursus': 'id_kursus',
        'idlevel': 'id_level'
    })

    # 3. CLEANING

    ## tipe harus sesuai FK
    df['id_kursus'] = df['id_kursus'].astype(str)
    df['id_level'] = pd.to_numeric(df['id_level'], errors='coerce')

    ## drop null (relasi ga boleh null)
    df = df.dropna(subset=['id_kursus', 'id_level'])

    ## convert int
    df['id_level'] = df['id_level'].astype(int)

    # 4. REMOVE DUPLICATE (relasi sering dobel)
    df = df.drop_duplicates(subset=['id_kursus', 'id_level'])

   # 5. NULL CHECK
    report_not_null_violations(df, "kursus_level")

    # 6. VALIDASI FK (opsional tapi bagus)
    # cek apakah id ada di tabel master
    df_kursus = fetch_df("SELECT id_kursus FROM kursus")
    df_level = fetch_df("SELECT id_level FROM level")

    df = df[df['id_kursus'].isin(df_kursus['id_kursus'])]
    df = df[df['id_level'].isin(df_level['id_level'])]

    # 7. PREVIEW
    preview_df(df, "SETELAH TRANSFORM")

    return df

In [37]:
def migrate_kursus_libur():
    print("\n=== MIGRATING KURSUS LIBUR ===")

    # 1. EXTRACT
    df_old = fetch_df("""
        SELECT idpendkursus, idlibur
        FROM libur_pendkursus
    """)

    preview_df(df_old, "DATA ASLI")

    # 2. TRANSFORM
    df = df_old.rename(columns={
        'idpendkursus': 'id_kursus',
        'idlibur': 'id_libur'
    })

    # 3. CLEANING

    df['id_kursus'] = df['id_kursus'].astype(str)
    df['id_libur'] = df['id_libur'].astype(str)

    ## drop null (FK ga boleh null)
    df = df.dropna(subset=['id_kursus', 'id_libur'])

    # 4. REMOVE DUPLICATE
    df = df.drop_duplicates(subset=['id_kursus', 'id_libur'])

    # 5. VALIDASI FK
    df_kursus = fetch_df("SELECT id_kursus FROM kursus")
    df_libur = fetch_df("SELECT id_libur FROM libur")

    df = df[df['id_kursus'].isin(df_kursus['id_kursus'])]
    df = df[df['id_libur'].isin(df_libur['id_libur'])]

    # 6. NULL CHECK
    report_nulls(df, "kursus_libur")

    # 7. PREVIEW
    preview_df(df, "SETELAH TRANSFORM")

    return df

## 5. Verifikasi Data

In [ ]:
# Verify data di DB baru
try:
    cursor_new.execute("SELECT COUNT(*) as count FROM [NEW_TABLE_NAME]")
    result = cursor_new.fetchone()
    count_new = result['count']
except:
    count_new = len(data_old)  # Fallback jika query gagal

print(f"Total records from old DB: {{len(data_old)}}")
print(f"Total records in new DB: {{count_new}}")

if count_new == len(data_old):
    print("✓ Verifikasi OK - Jumlah record cocok")
else:
    print(f"⚠ Warning - Perbedaan: {{abs(count_new - len(data_old))}} record")

Total records from old DB: {len(data_old)}
Total records in new DB: {count_new}
✓ Verifikasi OK - Jumlah record cocok


: 

: 

: 

## 6. Return Hasil Migrasi untuk migrate_db.py

## 7. Close Connection

In [ ]:
# Close semua koneksi database
try:
    cursor_old.close()
    cursor_new.close()
    db_old.close()
    db_new.close()
    print("✓ Database connections closed")
except:
    print("⚠ Error closing connections (mungkin sudah tertutup)")

: 

: 

: 